#### Step 1: Set up the schema for the table

In [0]:
USE CATALOG demo;
USE SCHEMA data_security;

#### Step 2: Create the table

In [0]:
CREATE OR REPLACE TABLE sales_abc (
    id INT,
    region STRING,
    email STRING,
    revenue INT
);

##### Step 3: Insert Sample Data

In [0]:
INSERT INTO sales_abc VALUES
    (1, 'UK', 'jane@sales.com', 100),
    (2, 'US', 'bob@sales.com', 200),
    (3, 'US', 'jane@sales.com', 300),
    (4, 'UK', 'bob@sales.com', 400),
    (5, 'US', 'jane@sales.com', 500),
    (6, 'UK', 'bob@sales.com', 600),
    (7, 'US', 'jane@sales.com', 700),
    (8, 'UK', 'bob@sales.com', 800),
    (9, 'US', 'jane@sales.com', 900),
    (10, 'UK', 'bob@sales.com', 1000)

In [0]:
SELECT * FROM sales_abc

#### Step 4: CREATE UDFs for row filter and column mask

##### Step 5: Create Governed Tags

1. `pii = email` (indicates the data contains personally identifiable information (PII))
2. `access_type = region` (Defines the type of access control to be applied to the data, such as region-based or department-based filtering) 

##### Step 6: Create Policy

(Could be at catalog/schema/table level)

1. policy_filter_region
2. policy_mask_email

In [0]:
CREATE OR REPLACE POLICY `policy_filter_region`
ON SCHEMA `demo`.`data_security`
COMMENT 'Applies row-level filtering based on region and user group membership to ensure user access only permitted regional data'
ROW FILTER `demo`.`data_security`.`fn_filter_region`
TO `us-sg`, `uk-sg`
FOR TABLES
MATCH COLUMNS has_tag_value('access_type','region') AS u0
USING COLUMNS (u0)

In [0]:
CREATE OR REPLACE POLICY `policy_mask_email`
ON SCHEMA `demo`.`data_security`
COMMENT 'Masks email data classified as PII based on user access priileges'
COLUMN MASK `demo`.`data_security`.`fn_mask_email`
TO `us-sg`, `uk-sg`
FOR TABLES
MATCH COLUMNS has_tag_value('pii','email') AS m
ON COLUMN m


##### Step 7: Atach tags to the table

1. sales_abac_region
2. sales_abac_email

In [0]:
select * from sales_abc;

#### Step 8: Create another table to domonstrate scalability of the ABAC implementation

In [0]:
CREATE OR REPLACE TABLE customers_abac (
    customer_id INT,
    customer_name STRING,
    email STRING,
    region STRING
);

In [0]:
ALTER TABLE customers_abac
ALTER COLUMN email
SET TAGS ('pii' = 'email')

In [0]:
ALTER TABLE customers_abac
ALTER COLUMN region
SET TAGS ('access_type' ='region')

In [0]:
INSERT INTO customers_abac VALUES
(1, 'John Doe', 'john.doe@example.com', 'US'),
(2, 'Jane Smith', 'jane.smith@example.com', 'UK'),
(3, 'Alice Johnson', 'alice.johnson@example.com', 'US'),
(4, 'Bob Brown', 'bob.brown@example.com', 'UK');

In [0]:
select * from customers_abac;